# Gold · Dimensión de Modo de Envío

## Propósito

Construir la dimensión de modos de envío de la capa Gold, para analizar entregas, lead time y variación de envío por tipo de transporte.

## Flujo

1. Obtener los valores distintos de modo de envío desde la capa Silver.
2. Asignar la llave sustituta y escribir la tabla.

## Destino

Capa Gold.

**Tabla:** `gld_dataco.supply_chain.desc_dim_shipping_mode`

**Llave sustituta:** `shipping_mode_key`

## Resultado esperado

La dimensión de modos de envío lista para relacionarse con los hechos de pedidos.

In [0]:
# ============================================================
# GLD_DIM_SHIPPING_MODE — Celda 1: Parámetros
# ============================================================

dbutils.widgets.text("silver_catalog", "slv_dataco", "Catálogo Silver")
dbutils.widgets.text("silver_schema", "supply_chain", "Esquema Silver")

dbutils.widgets.text("gold_catalog", "gld_dataco", "Catálogo Gold")
dbutils.widgets.text("gold_schema", "supply_chain", "Esquema Gold")

silver_catalog = dbutils.widgets.get("silver_catalog")
silver_schema = dbutils.widgets.get("silver_schema")
gold_catalog = dbutils.widgets.get("gold_catalog")
gold_schema = dbutils.widgets.get("gold_schema")

SILVER_PEDIDOS_FQN = f"{silver_catalog}.{silver_schema}.hd_pedidos"
GOLD_DIM_SHIPPING_MODE_FQN = f"{gold_catalog}.{gold_schema}.desc_dim_shipping_mode"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_catalog}.{gold_schema}")

print(f"✓ Origen: {SILVER_PEDIDOS_FQN}")
print(f"✓ Destino: {GOLD_DIM_SHIPPING_MODE_FQN}")

In [0]:
# ============================================================
# GLD_DIM_SHIPPING_MODE — Celda 2: INICIO DE LOG DE EJECUCIÓN
# ============================================================

import uuid

id_ejecucion = dbutils.jobs.taskValues.get(
    taskKey="prepare_pipeline_control",
    key="id_ejecucion",
    debugValue=""
)

LOG_ID = str(uuid.uuid4())
NOTEBOOK = "gld_dim_shipping_mode"
CAPA = "gold"

spark.sql(f"""
INSERT INTO metadata_dataco.supply_chain.hd_etl_log
(log_id, id_ejecucion, notebook, capa, estado, fecha_inicio)
VALUES ('{LOG_ID}', '{id_ejecucion}', '{NOTEBOOK}', '{CAPA}', 'EN_PROCESO', current_timestamp())
""")

dbutils.jobs.taskValues.set(key="log_id", value=LOG_ID)

print(f"✓ Registro EN_PROCESO creado en etl_log: {LOG_ID}")

In [0]:
# ============================================================
# GLD_DIM_SHIPPING_MODE — Celda 3: Valores distintos desde Silver
# ============================================================

from pyspark.sql import functions as F

df_shipping_mode = (
    spark.table(SILVER_PEDIDOS_FQN)
    .select("shipping_mode")
    .distinct()
    .filter(F.col("shipping_mode").isNotNull())
)

print(f"✓ Modos de envío distintos: {df_shipping_mode.count()}")
display(df_shipping_mode)

In [0]:
# ============================================================
# GLD_DIM_SHIPPING_MODE — Celda 4: Escritura con llave sustituta
# ============================================================

df_dim_shipping_mode = df_shipping_mode.withColumn(
    "shipping_mode_key", F.monotonically_increasing_id()
)

(
    df_dim_shipping_mode.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_DIM_SHIPPING_MODE_FQN)
)

print(f"✓ Tabla {GOLD_DIM_SHIPPING_MODE_FQN} escrita: {df_dim_shipping_mode.count()} filas")

In [0]:
# ============================================================
# GLD_DIM_SHIPPING_MODE — Celda 5: ACTUALIZACIÓN DEL LOG — EJECUCIÓN EXITOSA
# ============================================================

spark.sql(f"""
UPDATE metadata_dataco.supply_chain.hd_etl_log
SET estado = 'EXITOSO', fecha_fin = current_timestamp()
WHERE log_id = '{LOG_ID}'
""")

print("✓ etl_log actualizado: EXITOSO")